In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()



25/03/04 05:41:46 WARN Utils: Your hostname, codespaces-029a01 resolves to a loopback address: 127.0.0.1; using 10.0.1.231 instead (on interface eth0)
25/03/04 05:41:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/04 05:41:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.parquet('fhvhv/2021/01/')

In [4]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: integer (nullable = true)



In [7]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 00:23:13|2021-01-01 00:30:35|         147|         159|
|2021-01-06 11:43:12|2021-01-06 11:55:07|          79|         164|
|2021-01-04 15:35:32|2021-01-04 15:52:02|         174|          18|
|2021-01-04 13:42:15|2021-01-04 14:04:57|         201|         180|
|2021-01-03 18:42:03|2021-01-03 19:12:22|         132|          72|
|2021-01-01 01:51:18|2021-01-01 02:05:32|         174|         235|
|2021-01-05 10:20:54|2021-01-05 10:32:44|          35|          76|
|2021-01-04 12:34:52|2021-01-04 12:38:59|         231|          13|
|2021-01-02 20:12:56|2021-01-02 20:41:18|          87|         127|
|2021-01-02 15:14:38|2021-01-02 15:23:27|          11|          14|
|2021-01-04 12:40:42|2021-01-04 12:48:34|          83|         260|
|2021-01-02 20:22:51|2021-01-02 20:44:39|       

In [9]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime) ) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime) ) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-07|  2021-01-07|         142|         230|
| 2021-01-01|  2021-01-01|         133|          91|
| 2021-01-01|  2021-01-01|         147|         159|
| 2021-01-06|  2021-01-06|          79|         164|
| 2021-01-04|  2021-01-04|         174|          18|
| 2021-01-04|  2021-01-04|         201|         180|
| 2021-01-04|  2021-01-04|         230|         142|
| 2021-01-03|  2021-01-03|         132|          72|
| 2021-01-01|  2021-01-01|         188|          61|
| 2021-01-04|  2021-01-04|          97|         189|
| 2021-01-01|  2021-01-01|         174|         235|
| 2021-01-05|  2021-01-05|          35|          76|
| 2021-01-06|  2021-01-06|          35|          39|
| 2021-01-04|  2021-01-04|         231|          13|
| 2021-01-02|  2021-01-02|          87|         127|
| 2021-01-02|  2021-01-02|          17|       

In [31]:
def trip_duration(time_end, time_start):
    diff = (time_end-time_start).total_seconds()
    out_string = ""
    if diff > 3600:
        hours = int(diff / 3600)
        diff = diff % 3600
        out_string += f"{hours}h"
    if diff > 60:
        mins =int(diff / 60)
        diff = diff % 60
        out_string += f"{mins}m"
    if diff > 0:
        secs = diff
        out_string += f"{secs}s"
    return out_string
        

In [13]:
df.select('pickup_datetime').show()

+-------------------+
|    pickup_datetime|
+-------------------+
|2021-01-07 06:43:22|
|2021-01-01 16:01:26|
|2021-01-01 00:23:13|
|2021-01-06 11:43:12|
|2021-01-04 15:35:32|
|2021-01-04 13:42:15|
|2021-01-04 18:57:31|
|2021-01-03 18:42:03|
|2021-01-01 05:31:50|
|2021-01-04 20:21:47|
|2021-01-01 01:51:18|
|2021-01-05 10:20:54|
|2021-01-06 02:32:09|
|2021-01-04 12:34:52|
|2021-01-02 20:12:56|
|2021-01-02 16:55:48|
|2021-01-02 15:14:38|
|2021-01-01 05:54:50|
|2021-01-04 12:40:42|
|2021-01-01 14:58:57|
+-------------------+
only showing top 20 rows



In [15]:
from datetime import datetime

In [32]:
trip_duration(datetime(2021,1,7,6,45,22),datetime(2021,1,7,6,43,22))

'2m'